In [5]:
import pandas as pd
import numpy as np
from modulos.data_load import load, coins
from sklearn.preprocessing import MinMaxScaler

In [ ]:

# Armazenar resultados (opcional)
resultados = {}

for coin in coins:
    df = load(coin)
    print(f"{coin} →", df.columns.tolist())
    print(f'Processando moeda: {coin}')
    
    
    # 1. Limpam as colunas
    df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]
    
    # 2. Ordena por data
    df_moeda = df_moeda.sort_values('date').reset_index(drop=True)
    
    # 3. Cria features derivadas
    df_moeda['faixa_preco'] = df_moeda['high'] - df_moeda['low']
    df_moeda['retorno_pct_1d'] = df_moeda['close'].pct_change(1)
    df_moeda['retorno_pct_7d'] = df_moeda['close'].pct_change(7)
    df_moeda['momentum_7d'] = df_moeda['close'] - df_moeda['close'].shift(7)

    df_moeda['media_movel_7d'] = df_moeda['close'].rolling(window=7).mean()
    df_moeda['media_movel_14d'] = df_moeda['close'].rolling(window=14).mean()
    df_moeda['std_7d'] = df_moeda['close'].rolling(window=7).std()
    df_moeda['std_14d'] = df_moeda['close'].rolling(window=14).std()

    df_moeda['volume_btc_7d'] = df_moeda['volume_btc'].rolling(window=7).mean()
    df_moeda['taker_ratio'] = df_moeda['buytakeramount'] / df['volume_btc']
    df_moeda['buy_pressure'] = df_moeda['buytakerquantity'] / df_moeda['tradecount']
    df_moeda['volume_volatilidade_ratio'] = df_moeda['volume_usdd'] / df_moeda['faixa_preco']

    df_moeda['date'] = pd.to_datetime(df_moeda['date'])
    df_moeda['dia_da_semana'] = df_moeda['date'].dt.dayofweek
    
    # 4. Remove NaNs gerados por rolling
    df_moeda.dropna(inplace=True)
    
    # 5. Separa os conjuntos
    n = len(df_moeda)
    n_train = int(n * 0.7)
    n_val = int(n * 0.15)

    df_train = df_moeda.iloc[:n_train]
    df_val = df_moeda.iloc[n_train:n_train + n_val]
    df_test = df_moeda.iloc[n_train + n_val:]

    # 6. Normaliza (somente com treino!)
    features = [
        'close', 'media_movel_7d', 'media_movel_14d', 
        'std_7d', 'std_14d', 'momentum_7d', 'retorno_pct_1d', 'retorno_pct_7d',
        'volume_usd_7d', 'taker_ratio', 'buy_pressure', 'volume_volatilidade_ratio',
        'dia_da_semana'
    ]
    
    scaler = MinMaxScaler()
    scaler.fit(df_train[features])

    df_train[features] = scaler.transform(df_train[features])
    df_val[features] = scaler.transform(df_val[features])
    df_test[features] = scaler.transform(df_test[features])
    
    # 7. Salva para uso posterior (opcional)
    resultados[coin] = {
        'train': df_train,
        'val': df_val,
        'test': df_test,
        'scaler': scaler
    }

    print(f'{coin} - Train shape: {df_train.shape}, Val: {df_val.shape}, Test: {df_test.shape}')


AAVEBTC → ['unix', 'date', 'symbol', 'open', 'high', 'low', 'close', 'Volume AAVE', 'Volume BTC', 'buyTakerAmount', 'buyTakerQuantity', 'tradeCount', 'weightedAverage']
Processando moeda: AAVEBTC


KeyError: 'volume_usdd'